In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql import functions as F
import requests

In [0]:
env = dbutils.notebook.run("/Workspace/anihan_tech_np/anihan_tech_np/00_Parameters/env_dependencies", 120)
print(env)

In [0]:
df = spark.sql(f"select appcode, company_name, owner from anihan_tech_{env}.00_master_01_brz.masterdata_company")
display(df)

In [0]:
appcodes = [row["appcode"] for row in df.select("appcode").distinct().collect()]

dbutils.widgets.dropdown(
    "appcode",
    appcodes[0],      
    appcodes,
    "appcode"
)

selected_appcode = dbutils.widgets.get("appcode")
print(selected_appcode)

In [0]:
schemas = [
    f"00_{selected_appcode}_00_lz",
    f"00_{selected_appcode}_01_brz",
    f"00_{selected_appcode}_02_slv",
    f"00_{selected_appcode}_03_gld"
]

volumes = [
    "lz_vl", 
    "brz_vl", 
    "slv_vl", 
    "gld_vl"
]

for schema, volume in zip(schemas, volumes):
    spark.sql(f"drop volume if exists anihan_tech_{env}.{schema}.{volume}")

In [0]:
vsc = VectorSearchClient()

endpoint_name = f"{selected_appcode}-pdf-vector-endpoint"
index_name = f"anihan_tech_{env}.00_{selected_appcode}_01_brz.pdf_chunks_index"

vsc.delete_index(
    endpoint_name=endpoint_name,
    index_name=index_name
)

print(f"Deleted vector index: {index_name}")

In [0]:
host = spark.conf.get("spark.databricks.workspaceUrl")
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
url = f"https://{host}/api/2.0/vector-search/endpoints/{endpoint_name}"

response = requests.delete(
    url,
    headers={
        "Authorization": f"Bearer {token}"
    }
)

print(response.status_code)
print(response.text)

In [0]:
schema_list = ",".join([f"'{s}'" for s in schemas])

objects_df = spark.sql(f"""
select
    table_schema,
    table_name,
    table_type
from anihan_tech_{env}.information_schema.tables
where table_schema IN ({schema_list})
""")

display(objects_df)

for row in objects_df.collect():
    schema = row["table_schema"]
    name = row["table_name"]
    obj_type = row["table_type"]
    fq_name = f"anihan_tech_{env}.{schema}.{name}"

    print(f"Processing {fq_name} ({obj_type})")

    if obj_type == "FOREIGN":
        print(f"Skipping Vector Index object: {fq_name}")
        continue

    elif obj_type == "VIEW":
        spark.sql(f"drop view if exists {fq_name}")

    else:
        spark.sql(f"drop table if exists {fq_name}")

    print(f"Dropped {fq_name}")

In [0]:
for schema in schemas:
    spark.sql(f"drop schema if exists anihan_tech_{env}.{schema}")